# 📄 Markdown →  PDF Converter
### High-quality PDF generation with consistent styling, syntax highlighting & VS Code-like code blocks
---
> **How to use:**
> 1. Set your **file name** and paste your **Markdown content** in Cell below
> 2. Run all cells
> 3. Click the **Download** button to get your PDF

In [ ]:
# ============================================================
#  📝 CONFIGURATION — Set your file name & Markdown content here
# ============================================================

FILE_NAME = "test_pdf"  # PDF file name (without .pdf)

MD_CONTENT = r"""

# Product Catalog API - Complete Implementation Guide

**Version:** 1.0  
**Date:** February 15, 2026  
**Author:** Senior API Designer  
**For:** Junior Developer

---

## Table of Contents
1. [System Overview](#system-overview)
2. [Database Schema](#database-schema)
3. [API Endpoints](#api-endpoints)
4. [Data Transfer Objects](#data-transfer-objects)
5. [Business Rules](#business-rules)
6. [Error Handling](#error-handling)

---

## System Overview

### Purpose
A RESTful product catalog system for managing products and their categories. Products belong to categories (many-to-one relationship).

### Key Principles
- **Thin Controllers**: No business logic, just routing and response wrapping
- **Service Layer**: All business logic, validation, and orchestration
- **Repository Layer**: Database operations only
- **DTO Pattern**: Separate request/response objects from entities
- **Pagination**: List endpoints return paginated results
- **Summary vs Detail**: Lists return lightweight summaries, single-item endpoints return full details

---

## Database Schema

### Table: `categories`

| Column Name  | Type          | Constraints           | Description                          |
|--------------|---------------|-----------------------|--------------------------------------|
| id           | BIGINT        | PRIMARY KEY, AUTO     | Unique identifier                    |
| name         | VARCHAR(100)  | NOT NULL, UNIQUE      | Category name                        |
| description  | TEXT          | NULLABLE              | Category description                 |
| slug         | VARCHAR(120)  | NOT NULL, UNIQUE      | URL-friendly identifier              |
| created_at   | TIMESTAMP     | NOT NULL              | Record creation timestamp            |
| updated_at   | TIMESTAMP     | NOT NULL              | Last update timestamp                |

**Indexes:**
- PRIMARY KEY on `id`
- UNIQUE INDEX on `name`
- UNIQUE INDEX on `slug`

**Business Logic:**
- `slug` is auto-generated from `name` on creation: lowercase, spaces replaced with hyphens
- `created_at` and `updated_at` are auto-managed by the database/ORM

---

### Table: `products`

| Column Name     | Type            | Constraints              | Description                          |
|-----------------|-----------------|--------------------------|--------------------------------------|
| id              | BIGINT          | PRIMARY KEY, AUTO        | Unique identifier                    |
| name            | VARCHAR(255)    | NOT NULL                 | Product name                         |
| description     | TEXT            | NULLABLE                 | Product description                  |
| sku             | VARCHAR(50)     | NOT NULL, UNIQUE         | Stock Keeping Unit (business key)    |
| price           | DECIMAL(10,2)   | NOT NULL                 | Product price (must be positive)     |
| stock_quantity  | INTEGER         | NOT NULL, DEFAULT 0      | Available stock                      |
| status          | VARCHAR(20)     | NOT NULL, DEFAULT 'DRAFT'| Product lifecycle status             |
| category_id     | BIGINT          | NOT NULL, FOREIGN KEY    | References `categories(id)`          |
| image_url       | VARCHAR(500)    | NULLABLE                 | Product image URL                    |
| created_at      | TIMESTAMP       | NOT NULL                 | Record creation timestamp            |
| updated_at      | TIMESTAMP       | NOT NULL                 | Last update timestamp                |

**Foreign Keys:**
- `category_id` → `categories(id)` (ON DELETE CASCADE or RESTRICT based on business needs)

**Indexes:**
- PRIMARY KEY on `id`
- UNIQUE INDEX on `sku`
- INDEX on `status`
- INDEX on `category_id`
- INDEX on `name`

**Business Logic:**
- `status` enum values: `DRAFT`, `ACTIVE`, `INACTIVE`, `OUT_OF_STOCK`
- `created_at` and `updated_at` are auto-managed
- `stock_quantity` must be >= 0
- `price` must be > 0

---

## API Endpoints

### Base URL
```
/products
/categories
```

---

## Category Endpoints

### 1. Create Category

**Endpoint:** `POST /categories`

**Description:** Creates a new category with auto-generated slug.

**Request Headers:**
```
Content-Type: application/json
```

**Request Body:**
```json
{
  "name": "Electronics",
  "description": "Electronic devices and gadgets"
}
```

**Request Validation Rules:**
- `name`: REQUIRED, 2-100 characters
- `description`: OPTIONAL, max 2000 characters

**Success Response: 201 Created**
```json
{
  "success": true,
  "message": "Category created successfully",
  "data": {
    "id": 1,
    "name": "Electronics",
    "description": "Electronic devices and gadgets",
    "slug": "electronics",
    "productCount": 0,
    "createdAt": "2026-02-15T10:30:00",
    "updatedAt": "2026-02-15T10:30:00"
  }
}
```

**Error Responses:**

400 Bad Request - Validation failure:
```json
{
  "success": false,
  "message": "Validation failed",
  "errors": [
    {
      "field": "name",
      "message": "Category name is required"
    }
  ]
}
```

409 Conflict - Duplicate name:
```json
{
  "success": false,
  "message": "Category with name 'Electronics' already exists"
}
```

---

### 2. Get All Categories (Paginated)

**Endpoint:** `GET /categories`

**Description:** Retrieves all categories with pagination.

**Query Parameters:**

| Parameter | Type | Required | Default | Description              |
|-----------|------|----------|---------|--------------------------|
| page      | int  | No       | 0       | Page number (0-based)    |
| size      | int  | No       | 10      | Items per page           |

**Example URLs:**
```
GET /categories
GET /categories?page=0&size=20
GET /categories?page=2
```

**Success Response: 200 OK**
```json
{
  "success": true,
  "data": {
    "content": [
      {
        "id": 1,
        "name": "Electronics",
        "description": "Electronic devices and gadgets",
        "slug": "electronics",
        "productCount": 25,
        "createdAt": "2026-02-15T10:30:00",
        "updatedAt": "2026-02-15T10:30:00"
      }
    ],
    "page": 0,
    "size": 10,
    "totalElements": 15,
    "totalPages": 2,
    "hasNext": true,
    "hasPrevious": false
  }
}
```

---

### 3. Get Category by ID

**Endpoint:** `GET /categories/{id}`

**Description:** Retrieves a single category by its ID.

**Path Parameters:**
- `id`: Category ID (Long)

**Success Response: 200 OK**
```json
{
  "success": true,
  "data": {
    "id": 1,
    "name": "Electronics",
    "description": "Electronic devices and gadgets",
    "slug": "electronics",
    "productCount": 25,
    "createdAt": "2026-02-15T10:30:00",
    "updatedAt": "2026-02-15T10:30:00"
  }
}
```

**Error Response: 404 Not Found**
```json
{
  "success": false,
  "message": "Category not found with id: 999"
}
```

---

### 4. Update Category (Partial)

**Endpoint:** `PATCH /categories/{id}`

**Description:** Updates a category. Only fields provided in request are updated (partial update).

**Path Parameters:**
- `id`: Category ID (Long)

**Request Body:**
```json
{
  "name": "Consumer Electronics",
  "description": "Updated description"
}
```

**Validation Rules:**
- `name`: OPTIONAL, if provided: 2-100 characters
- `description`: OPTIONAL, if provided: max 2000 characters
- Only non-null fields are updated

**Success Response: 200 OK**
```json
{
  "success": true,
  "message": "Category updated successfully",
  "data": {
    "id": 1,
    "name": "Consumer Electronics",
    "description": "Updated description",
    "slug": "electronics",
    "productCount": 25,
    "createdAt": "2026-02-15T10:30:00",
    "updatedAt": "2026-02-15T14:45:00"
  }
}
```

**Error Responses:**
- 404: Category not found
- 400: Validation failure
- 409: Duplicate name

---

### 5. Delete Category

**Endpoint:** `DELETE /categories/{id}`

**Description:** Deletes a category. Decide business rule: prevent deletion if products exist OR cascade delete OR set products' category to null.

**Path Parameters:**
- `id`: Category ID (Long)

**Success Response: 200 OK**
```json
{
  "success": true,
  "message": "Category deleted successfully"
}
```

**Error Response: 404 Not Found**
```json
{
  "success": false,
  "message": "Category not found with id: 999"
}
```

---

## Product Endpoints

### 6. Create Product

**Endpoint:** `POST /products`

**Description:** Creates a new product.

**Request Body:**
```json
{
  "name": "iPhone 15 Pro",
  "description": "Latest flagship smartphone with A17 Pro chip",
  "sku": "IPHONE-15-PRO-256",
  "price": 999.99,
  "stockQuantity": 50,
  "categoryId": 1,
  "imageUrl": "https://example.com/images/iphone15pro.jpg"
}
```

**Validation Rules:**
- `name`: REQUIRED, 2-255 characters
- `description`: OPTIONAL, max 5000 characters
- `sku`: REQUIRED, 3-50 characters, must be unique
- `price`: REQUIRED, must be > 0
- `categoryId`: REQUIRED, must reference existing category
- `stockQuantity`: OPTIONAL, default 0, must be >= 0
- `imageUrl`: OPTIONAL, max 500 characters

**Success Response: 201 Created**
```json
{
  "success": true,
  "message": "Product created successfully",
  "data": {
    "id": 101,
    "name": "iPhone 15 Pro",
    "description": "Latest flagship smartphone with A17 Pro chip",
    "sku": "IPHONE-15-PRO-256",
    "price": 999.99,
    "stockQuantity": 50,
    "status": "DRAFT",
    "imageUrl": "https://example.com/images/iphone15pro.jpg",
    "category": {
      "id": 1,
      "name": "Electronics",
      "description": "Electronic devices and gadgets",
      "slug": "electronics",
      "productCount": 26,
      "createdAt": "2026-02-15T10:30:00",
      "updatedAt": "2026-02-15T10:30:00"
    },
    "createdAt": "2026-02-15T11:20:00",
    "updatedAt": "2026-02-15T11:20:00"
  }
}
```

**Error Responses:**

400 Bad Request - Validation:
```json
{
  "success": false,
  "message": "Validation failed",
  "errors": [
    {
      "field": "price",
      "message": "Price must be greater than zero"
    }
  ]
}
```

404 Not Found - Category doesn't exist:
```json
{
  "success": false,
  "message": "Category not found with id: 999"
}
```

409 Conflict - Duplicate SKU:
```json
{
  "success": false,
  "message": "Product with SKU 'IPHONE-15-PRO-256' already exists"
}
```

---

### 7. Get Product by ID (Full Detail)

**Endpoint:** `GET /products/{id}`

**Description:** Retrieves full product details including nested category object.

**Path Parameters:**
- `id`: Product ID (Long)

**Success Response: 200 OK**
```json
{
  "success": true,
  "data": {
    "id": 101,
    "name": "iPhone 15 Pro",
    "description": "Latest flagship smartphone with A17 Pro chip",
    "sku": "IPHONE-15-PRO-256",
    "price": 999.99,
    "stockQuantity": 50,
    "status": "ACTIVE",
    "imageUrl": "https://example.com/images/iphone15pro.jpg",
    "category": {
      "id": 1,
      "name": "Electronics",
      "description": "Electronic devices and gadgets",
      "slug": "electronics",
      "productCount": 26,
      "createdAt": "2026-02-15T10:30:00",
      "updatedAt": "2026-02-15T10:30:00"
    },
    "createdAt": "2026-02-15T11:20:00",
    "updatedAt": "2026-02-15T11:20:00"
  }
}
```

**Why Full Detail?**  
This is a single-item endpoint. The client wants everything: description, timestamps, full category object. Used when displaying a product detail page.

**Error Response: 404 Not Found**
```json
{
  "success": false,
  "message": "Product not found with id: 999"
}
```

---

### 8. Get All Products (Paginated Summary)

**Endpoint:** `GET /products`

**Description:** Retrieves paginated list of products with lightweight summaries.

**Query Parameters:**

| Parameter  | Type | Required | Default | Description              |
|------------|------|----------|---------|--------------------------|
| page       | int  | No       | 0       | Page number (0-based)    |
| size       | int  | No       | 10      | Items per page           |
| categoryId | Long | No       | -       | Filter by category       |

**Example URLs:**
```
GET /products
GET /products?page=0&size=20
GET /products?categoryId=1
GET /products?categoryId=1&page=2&size=5
```

**Success Response: 200 OK**
```json
{
  "success": true,
  "data": {
    "content": [
      {
        "id": 101,
        "name": "iPhone 15 Pro",
        "sku": "IPHONE-15-PRO-256",
        "price": 999.99,
        "status": "ACTIVE",
        "stockQuantity": 50,
        "imageUrl": "https://example.com/images/iphone15pro.jpg",
        "categoryName": "Electronics"
      },
      {
        "id": 102,
        "name": "MacBook Pro 16",
        "sku": "MBP-16-M3-512",
        "price": 2499.99,
        "status": "ACTIVE",
        "stockQuantity": 15,
        "imageUrl": "https://example.com/images/macbook.jpg",
        "categoryName": "Electronics"
      }
    ],
    "page": 0,
    "size": 10,
    "totalElements": 125,
    "totalPages": 13,
    "hasNext": true,
    "hasPrevious": false
  }
}
```

**Why Summary Instead of Full?**  
List endpoints return lightweight objects. No description, no timestamps, no full category object—just category name. This reduces payload size and improves performance when listing 50-100+ products.

---

### 9. Search Products

**Endpoint:** `GET /products/search`

**Description:** Searches products by name (case-insensitive, partial match). Returns lightweight summaries.

**Query Parameters:**

| Parameter | Type   | Required | Default | Description              |
|-----------|--------|----------|---------|--------------------------|
| query     | string | Yes      | -       | Search term              |
| page      | int    | No       | 0       | Page number              |
| size      | int    | No       | 10      | Items per page           |

**Example URLs:**
```
GET /products/search?query=phone
GET /products/search?query=MacBook&page=0&size=5
```

**Search Logic:**  
Case-insensitive LIKE match on product name (e.g., `WHERE LOWER(name) LIKE '%phone%'`)

**Success Response: 200 OK**
```json
{
  "success": true,
  "data": {
    "content": [
      {
        "id": 101,
        "name": "iPhone 15 Pro",
        "sku": "IPHONE-15-PRO-256",
        "price": 999.99,
        "status": "ACTIVE",
        "stockQuantity": 50,
        "imageUrl": "https://example.com/images/iphone15pro.jpg",
        "categoryName": "Electronics"
      }
    ],
    "page": 0,
    "size": 10,
    "totalElements": 1,
    "totalPages": 1,
    "hasNext": false,
    "hasPrevious": false
  }
}
```

**Error Response: 400 Bad Request**  
If `query` parameter is missing.

---

### 10. Update Product (Partial)

**Endpoint:** `PATCH /products/{id}`

**Description:** Updates a product. Only fields provided in request are updated.

**Path Parameters:**
- `id`: Product ID (Long)

**Request Body:**  
All fields optional. Only non-null fields will update the entity.

```json
{
  "price": 899.99,
  "stockQuantity": 75,
  "status": "ACTIVE"
}
```

**Validation Rules:**
- `name`: OPTIONAL, if provided: 2-255 characters
- `description`: OPTIONAL, if provided: max 5000 characters
- `price`: OPTIONAL, if provided: must be > 0
- `categoryId`: OPTIONAL, if provided: must reference existing category
- `stockQuantity`: OPTIONAL, if provided: must be >= 0
- `status`: OPTIONAL, if provided: must be one of: DRAFT, ACTIVE, INACTIVE, OUT_OF_STOCK
- `imageUrl`: OPTIONAL, if provided: max 500 characters

**Success Response: 200 OK**
```json
{
  "success": true,
  "message": "Product updated successfully",
  "data": {
    "id": 101,
    "name": "iPhone 15 Pro",
    "description": "Latest flagship smartphone with A17 Pro chip",
    "sku": "IPHONE-15-PRO-256",
    "price": 899.99,
    "stockQuantity": 75,
    "status": "ACTIVE",
    "imageUrl": "https://example.com/images/iphone15pro.jpg",
    "category": {
      "id": 1,
      "name": "Electronics",
      "description": "Electronic devices and gadgets",
      "slug": "electronics",
      "productCount": 26,
      "createdAt": "2026-02-15T10:30:00",
      "updatedAt": "2026-02-15T10:30:00"
    },
    "createdAt": "2026-02-15T11:20:00",
    "updatedAt": "2026-02-15T16:45:00"
  }
}
```

**Error Responses:**
- 404: Product not found
- 400: Validation failure
- 404: Category not found (if categoryId provided)

---

### 11. Delete Product

**Endpoint:** `DELETE /products/{id}`

**Description:** Permanently deletes a product.

**Path Parameters:**
- `id`: Product ID (Long)

**Success Response: 200 OK**
```json
{
  "success": true,
  "message": "Product deleted successfully"
}
```

**Error Response: 404 Not Found**
```json
{
  "success": false,
  "message": "Product not found with id: 999"
}
```

**Alternative Implementation:**  
Instead of hard delete, set `status = INACTIVE` (soft delete pattern). Good for auditing and data recovery.

---

## Data Transfer Objects

### Request DTOs

#### CreateCategoryRequest
```
name:        string  [REQUIRED, 2-100 chars]
description: string  [OPTIONAL, max 2000 chars]
```

#### UpdateCategoryRequest
```
name:        string  [OPTIONAL, 2-100 chars]
description: string  [OPTIONAL, max 2000 chars]
```

#### CreateProductRequest
```
name:          string  [REQUIRED, 2-255 chars]
description:   string  [OPTIONAL, max 5000 chars]
sku:           string  [REQUIRED, 3-50 chars, unique]
price:         decimal [REQUIRED, > 0]
stockQuantity: integer [OPTIONAL, >= 0, default: 0]
categoryId:    long    [REQUIRED, must exist]
imageUrl:      string  [OPTIONAL, max 500 chars]
```

#### UpdateProductRequest
```
name:          string  [OPTIONAL, 2-255 chars]
description:   string  [OPTIONAL, max 5000 chars]
price:         decimal [OPTIONAL, > 0]
stockQuantity: integer [OPTIONAL, >= 0]
categoryId:    long    [OPTIONAL, must exist]
status:        string  [OPTIONAL, enum: DRAFT|ACTIVE|INACTIVE|OUT_OF_STOCK]
imageUrl:      string  [OPTIONAL, max 500 chars]
```

---

### Response DTOs

#### CategoryResponse
```json
{
  "id": long,
  "name": string,
  "description": string,
  "slug": string,
  "productCount": integer,
  "createdAt": timestamp,
  "updatedAt": timestamp
}
```

**Usage:**
- GET /categories/{id}
- POST /categories
- PATCH /categories/{id}
- Nested in ProductResponse

---

#### ProductResponse (Full Detail)
```json
{
  "id": long,
  "name": string,
  "description": string,
  "sku": string,
  "price": decimal,
  "stockQuantity": integer,
  "status": string,
  "imageUrl": string,
  "category": CategoryResponse,  // Nested full category object
  "createdAt": timestamp,
  "updatedAt": timestamp
}
```

**Usage:**
- GET /products/{id}
- POST /products
- PATCH /products/{id}

**Why nested category object?**  
Avoids N+1 API calls. Client gets product + category in one request.

---

#### ProductSummaryResponse (Lightweight)
```json
{
  "id": long,
  "name": string,
  "sku": string,
  "price": decimal,
  "status": string,
  "stockQuantity": integer,
  "imageUrl": string,
  "categoryName": string  // Just the name, not full object
}
```

**Usage:**
- GET /products (list)
- GET /products/search

**Why lightweight?**  
Used in lists. No description, no timestamps, no full category object. Optimized for performance and bandwidth.

---

#### PageResponse<T>
```json
{
  "content": [T],        // Array of items
  "page": integer,       // Current page (0-based)
  "size": integer,       // Items per page
  "totalElements": long, // Total items across all pages
  "totalPages": integer, // Total pages
  "hasNext": boolean,    // Has next page
  "hasPrevious": boolean // Has previous page
}
```

**Generic wrapper for paginated lists.**

---

#### ApiResponse<T>
```json
{
  "success": boolean,    // true or false
  "message": string,     // Optional message
  "data": T,             // Response payload
  "errors": [            // Optional validation errors
    {
      "field": string,
      "message": string
    }
  ]
}
```

**Standard wrapper for all responses.**

---

## Business Rules

### Product Status Lifecycle

**Status Enum:** DRAFT → ACTIVE → INACTIVE | OUT_OF_STOCK

1. **DRAFT**: Product created, not yet available for sale
2. **ACTIVE**: Product is live and available
3. **INACTIVE**: Product soft-deleted or disabled
4. **OUT_OF_STOCK**: Automatically set when stockQuantity reaches 0

**Automatic Transitions:**
- When `stockQuantity` becomes 0 → set status to `OUT_OF_STOCK`
- When stock is added to `OUT_OF_STOCK` product → set status to `ACTIVE`

**Manual Transitions (via PATCH):**
- DRAFT → ACTIVE (activate product)
- ACTIVE → INACTIVE (deactivate product)
- Any status can be set manually via PATCH /products/{id}

---

### Category Slug Generation

When creating a category:
1. If `slug` is not provided, generate it from `name`
2. Convert to lowercase
3. Replace spaces with hyphens
4. Remove special characters

**Example:**
- Name: "Consumer Electronics"
- Auto-generated slug: "consumer-electronics"

---

### Validation Rules Summary

**Category:**
- Name: 2-100 chars, unique, required
- Description: 0-2000 chars, optional

**Product:**
- Name: 2-255 chars, required
- Description: 0-5000 chars, optional
- SKU: 3-50 chars, unique, required
- Price: > 0, required
- Stock: >= 0, default 0
- Category: must exist, required
- ImageUrl: 0-500 chars, optional

---

## Error Handling

### HTTP Status Codes

| Code | Meaning              | When to Use                                  |
|------|----------------------|----------------------------------------------|
| 200  | OK                   | Successful GET, PATCH, DELETE                |
| 201  | Created              | Successful POST (resource created)           |
| 400  | Bad Request          | Validation failure, malformed request        |
| 404  | Not Found            | Resource doesn't exist                       |
| 409  | Conflict             | Duplicate resource (name, SKU)               |
| 500  | Internal Server Error| Unexpected server error                      |

---

### Error Response Format

**Validation Error (400):**
```json
{
  "success": false,
  "message": "Validation failed",
  "errors": [
    {
      "field": "name",
      "message": "Product name is required"
    },
    {
      "field": "price",
      "message": "Price must be greater than zero"
    }
  ]
}
```

**Not Found (404):**
```json
{
  "success": false,
  "message": "Product not found with id: 999"
}
```

**Conflict (409):**
```json
{
  "success": false,
  "message": "Product with SKU 'IPHONE-15-PRO-256' already exists"
}
```

---

## Implementation Architecture

### Layer Responsibilities

#### 1. Controller Layer
**Responsibilities:**
- Define HTTP endpoint (method + path)
- Extract request parameters (@PathVariable, @RequestParam, @RequestBody)
- Trigger validation (@Valid)
- Delegate to service layer
- Wrap response in ApiResponse
- Return correct HTTP status code

**NO business logic in controllers!**

---

#### 2. Service Layer (Interface)
**Define method signatures:**
```
CategoryResponse createCategory(CreateCategoryRequest request)
CategoryResponse getCategoryById(Long id)
PageResponse<CategoryResponse> getAllCategories(int page, int size)
CategoryResponse updateCategory(Long id, UpdateCategoryRequest request)
void deleteCategory(Long id)

ProductResponse createProduct(CreateProductRequest request)
ProductResponse getProductById(Long id)
PageResponse<ProductSummaryResponse> getAllProducts(int page, int size)
PageResponse<ProductSummaryResponse> getProductsByCategory(Long categoryId, int page, int size)
PageResponse<ProductSummaryResponse> searchProducts(String query, int page, int size)
ProductResponse updateProduct(Long id, UpdateProductRequest request)
void deleteProduct(Long id)
```

---

#### 3. Service Implementation Layer
**Responsibilities:**
- All business logic
- Validation beyond @Valid (uniqueness, existence checks)
- Orchestration (coordinate multiple repository calls)
- Exception throwing (ResourceNotFoundException, DuplicateResourceException)
- Entity-to-DTO mapping

**Implementation Steps for Create Product:**
1. Check if SKU already exists → throw DuplicateResourceException (409)
2. Check if category exists → throw ResourceNotFoundException (404)
3. Map CreateProductRequest → Product entity
4. Set default status = DRAFT
5. Save entity via repository
6. Map Product entity → ProductResponse
7. Return ProductResponse

---

#### 4. Repository Layer (Interface)
**Database operations only. Example methods:**
```
// Standard CRUD
Product save(Product entity)
Optional<Product> findById(Long id)
Page<Product> findAll(Pageable pageable)
void deleteById(Long id)
boolean existsById(Long id)

// Custom queries
Optional<Product> findBySku(String sku)
Page<Product> findByCategoryId(Long categoryId, Pageable pageable)
Page<Product> findByNameContainingIgnoreCase(String query, Pageable pageable)
boolean existsBySku(String sku)

Optional<Category> findByName(String name)
boolean existsByName(String name)
```

---

#### 5. Mapper Layer
**Converts between entities and DTOs. Use MapStruct or manual mapping.**

**Methods needed:**
```
// Category
CategoryResponse toCategoryResponse(Category entity)
Category toCategory(CreateCategoryRequest request)
void updateCategoryFromRequest(UpdateCategoryRequest request, Category entity)

// Product
ProductResponse toProductResponse(Product entity)
ProductSummaryResponse toProductSummaryResponse(Product entity)
Product toProduct(CreateProductRequest request)
void updateProductFromRequest(UpdateProductRequest request, Product entity)
```

**Mapping Rules:**
- For nested objects: Map category entity → CategoryResponse when building ProductResponse
- For partial updates: Only update fields that are non-null in the request DTO
- Auto-generate timestamps in entity @PrePersist and @PreUpdate

---

## Testing Checklist

### Category Endpoints
- [ ] Create category with valid data → 201
- [ ] Create category with duplicate name → 409
- [ ] Create category with invalid name (too short) → 400
- [ ] Get category by valid ID → 200
- [ ] Get category by invalid ID → 404
- [ ] Get all categories with pagination → 200
- [ ] Update category with valid data → 200
- [ ] Update category with duplicate name → 409
- [ ] Delete category → 200
- [ ] Delete non-existent category → 404

### Product Endpoints
- [ ] Create product with valid data → 201
- [ ] Create product with duplicate SKU → 409
- [ ] Create product with invalid category → 404
- [ ] Create product with negative price → 400
- [ ] Get product by valid ID → 200 with full details
- [ ] Get product by invalid ID → 404
- [ ] Get all products → 200 with summaries
- [ ] Filter products by category → 200
- [ ] Search products by name → 200
- [ ] Update product with valid data → 200
- [ ] Update product with invalid category → 404
- [ ] Delete product → 200
- [ ] Delete non-existent product → 404

---

## Quick Reference

### HTTP Methods
- **POST**: Create new resource → 201 Created
- **GET**: Retrieve resource(s) → 200 OK
- **PATCH**: Partial update → 200 OK
- **DELETE**: Remove resource → 200 OK (or 204 No Content)

### When to Return What
- **Create/Update/Get Detail**: Full DTO (ProductResponse with nested CategoryResponse)
- **List/Search**: Summary DTO (ProductSummaryResponse with just category name)
- **Delete**: Success message only (or empty body with 204)

### Pagination Parameters
- `page`: 0-based page number, default 0
- `size`: items per page, default 10

### Unique Constraints
- Category: `name`, `slug`
- Product: `sku`

---

## Final Notes

**This document is framework-agnostic.** You can implement this in:
- Spring Boot (Java)
- Django/FastAPI (Python)
- Express/NestJS (Node.js)
- Laravel (PHP)
- ASP.NET Core (C#)

**Core principles remain the same:**
1. Separate entities from DTOs
2. Thin controllers, fat services
3. Validate at boundaries
4. Use summary DTOs for lists, full DTOs for details
5. Return correct HTTP status codes
6. Wrap responses in standard format

**Next Steps:**
1. Set up database with these tables
2. Create entity classes
3. Create DTOs (request/response)
4. Create repositories
5. Create mappers
6. Implement services
7. Implement controllers
8. Test each endpoint

Good luck! 🚀


"""

In [ ]:
# ============================================================
#  📦 Install Dependencies
# ============================================================
import subprocess, sys

packages = ["markdown", "Pygments", "weasyprint"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All dependencies installed successfully!")

In [ ]:
# ============================================================
#  🎨 PDF Styling Engine + Code Highlighter
# ============================================================

import markdown
from markdown.extensions import Extension
from markdown.treeprocessors import Treeprocessor
from markdown.preprocessors import Preprocessor
from pygments import highlight as pyg_highlight
from pygments.lexers import get_lexer_by_name, guess_lexer, TextLexer
from pygments.formatters import HtmlFormatter
import re
import xml.etree.ElementTree as ET


# ============================================================
#  Extension 0: Auto-insert blank lines before list starts
#  Fixes: "- item" right after "**Bold:**" or "### Heading"
#  without a blank line → markdown treats it as paragraph text
# ============================================================
class ListSpacingPreprocessor(Preprocessor):
    LIST_RE = re.compile(r'^(\s*)([-*+]|\d+[.)]) ')

    def run(self, lines):
        result = []
        for i, line in enumerate(lines):
            if self.LIST_RE.match(line):
                # Check if previous non-blank line is NOT a list item
                prev_line = ''
                for j in range(len(result) - 1, -1, -1):
                    if result[j].strip():
                        prev_line = result[j]
                        break
                if prev_line and not self.LIST_RE.match(prev_line):
                    # Insert blank line before this list start
                    result.append('')
            result.append(line)
        return result


class ListSpacingExtension(Extension):
    def extendMarkdown(self, md):
        md.preprocessors.register(
            ListSpacingPreprocessor(md), 'list_spacing', 120
        )


# ============================================================
#  Extension 1: Yellow highlight ONLY on H2 question text
# ============================================================
class QuestionHighlighter(Treeprocessor):
    HIGHLIGHT_STYLE = (
        'background-color: #ffff00; '
        'padding: 2px 8px; '
        'border-radius: 4px; '
        'font-weight: 700; '
        'box-decoration-break: clone; '
        '-webkit-box-decoration-break: clone;'
    )

    def run(self, root):
        for element in root.iter('h2'):
            span = ET.Element('span')
            span.set('style', self.HIGHLIGHT_STYLE)
            span.text = element.text or ''
            element.text = None
            children = list(element)
            for child in children:
                element.remove(child)
                span.append(child)
            element.append(span)


class QuestionHighlightExtension(Extension):
    def extendMarkdown(self, md):
        md.treeprocessors.register(QuestionHighlighter(md), 'question_highlight', 5)


# ============================================================
#  Extension 2: Inline styles on ALL table elements
# ============================================================
class TableStyler(Treeprocessor):
    TABLE_STYLE = (
        'width: 100%; border-collapse: collapse; margin: 14px 0; '
        'font-size: 10.5pt; page-break-inside: avoid; '
        'border: 2px solid #b8960f;'
    )
    TH_STYLE = (
        'background-color: #e6d270; color: #1a1206; font-weight: 700; '
        'padding: 10px 14px; border: 1px solid #b8960f; text-align: left; '
        'font-size: 10.5pt;'
    )
    TD_EVEN = (
        'padding: 9px 14px; border: 1px solid #d4c090; '
        'background-color: #fff8e1; color: #2c2417; font-size: 10.5pt;'
    )
    TD_ODD = (
        'padding: 9px 14px; border: 1px solid #d4c090; '
        'background-color: #ffedaa; color: #2c2417; font-size: 10.5pt;'
    )

    def run(self, root):
        for table in root.iter('table'):
            table.set('style', self.TABLE_STYLE)
        for th in root.iter('th'):
            th.set('style', self.TH_STYLE)
        for tbody in root.iter('tbody'):
            for i, tr in enumerate(c for c in tbody if c.tag == 'tr'):
                style = self.TD_EVEN if i % 2 == 0 else self.TD_ODD
                for td in (c for c in tr if c.tag == 'td'):
                    td.set('style', style)
        for table in root.iter('table'):
            for i, tr in enumerate(c for c in table if c.tag == 'tr'):
                style = self.TD_EVEN if i % 2 == 0 else self.TD_ODD
                for td in (c for c in tr if c.tag == 'td'):
                    td.set('style', style)


class TableStyleExtension(Extension):
    def extendMarkdown(self, md):
        md.treeprocessors.register(TableStyler(md), 'table_styler', 4)


# ============================================================
#  Extension 3: Fenced code → Pygments INLINE styles
#  FIXED: non-greedy, preserves surrounding blank lines for MD
# ============================================================
class InlineCodeHighlightPreprocessor(Preprocessor):
    # Match fenced blocks: ```lang\ncode\n```
    # Non-greedy, line-anchored, preserves blank lines
    FENCED_RE = re.compile(
        r'^```([\w+-]*)[ \t]*\n(.*?\n)```[ \t]*$',
        re.MULTILINE | re.DOTALL
    )

    PRE_STYLE = (
        'background-color: #1e1e1e; '
        'color: #d4d4d4; '
        'border-radius: 8px; '
        'padding: 16px 20px; '
        'font-family: Consolas, monospace; '
        'font-size: 10.5pt; '
        'line-height: 1.7; '
        'border-left: 4px solid #007acc; '
        'margin: 12px 0 18px 0; '
        'white-space: pre-wrap; '
        'word-wrap: break-word;'
    )

    def run(self, lines):
        text = '\n'.join(lines)

        # Use a non-greedy approach: find each ``` pair one at a time
        result = []
        last_end = 0

        for match in self.FENCED_RE.finditer(text):
            # Add text before this match
            result.append(text[last_end:match.start()])

            lang = match.group(1).strip() or ''
            code = match.group(2)
            if code.endswith('\n'):
                code = code[:-1]

            # Pick lexer
            try:
                lexer = get_lexer_by_name(lang) if lang else TextLexer()
            except Exception:
                try:
                    lexer = guess_lexer(code)
                except Exception:
                    lexer = TextLexer()

            # Highlight with inline styles
            formatter = HtmlFormatter(
                noclasses=True,
                style='monokai',
                wrapcode=True,
                linenos=False,
            )
            highlighted = pyg_highlight(code, lexer, formatter)

            # Inject inline pre styles
            highlighted = highlighted.replace(
                '<pre',
                f'<pre style="{self.PRE_STYLE}"',
                1
            )

            # Wrap with blank lines to preserve markdown block separation
            result.append(f'\n\n{highlighted}\n\n')
            last_end = match.end()

        # Add remaining text
        result.append(text[last_end:])

        return ''.join(result).split('\n')


class InlineCodeHighlightExtension(Extension):
    def extendMarkdown(self, md):
        md.preprocessors.register(
            InlineCodeHighlightPreprocessor(md), 'inline_code_highlight', 110
        )


# ============================================================
#  Extension 4: Inline styles on list items (ul/ol/li)
#  Forces WeasyPrint to render lists properly as block elements
# ============================================================
class ListStyler(Treeprocessor):
    UL_STYLE = (
        'margin: 8px 0 14px 0; padding-left: 28px; '
        'list-style-type: disc; color: #2c2417;'
    )
    OL_STYLE = (
        'margin: 8px 0 14px 0; padding-left: 28px; '
        'list-style-type: decimal; color: #2c2417;'
    )
    LI_STYLE = (
        'display: list-item; margin-bottom: 5px; '
        'line-height: 1.7; color: #2c2417;'
    )

    def run(self, root):
        for ul in root.iter('ul'):
            ul.set('style', self.UL_STYLE)
        for ol in root.iter('ol'):
            ol.set('style', self.OL_STYLE)
        for li in root.iter('li'):
            li.set('style', self.LI_STYLE)


class ListStyleExtension(Extension):
    def extendMarkdown(self, md):
        md.treeprocessors.register(ListStyler(md), 'list_styler', 3)


# ============================================================
#  PDF CSS
# ============================================================
PDF_CSS = """
@page {
    size: A4;
    margin: 25mm 20mm 30mm 20mm;
    background-color: #fff2cc;

    @bottom-center {
        content: "Page " counter(page) " of " counter(pages);
        font-family: Arial, sans-serif;
        font-size: 8pt;
        color: #8b7d6b;
    }
}

body {
    font-family: Arial, 'Segoe UI', Helvetica, sans-serif;
    font-size: 11pt;
    line-height: 1.75;
    color: #2c2417;
    background-color: #fff2cc;
    margin: 0;
    padding: 0;
}

h1 {
    font-family: Georgia, 'Times New Roman', serif;
    font-size: 22pt;
    font-weight: 900;
    text-align: center;
    color: #1a1206;
    background-color: #ffe680;
    border: 3px solid #c4a84a;
    border-radius: 12px;
    padding: 18px 24px;
    margin: 0 0 28px 0;
    letter-spacing: 0.5px;
    page-break-after: avoid;
}

h2 {
    font-family: Arial, sans-serif;
    font-size: 12.5pt;
    font-weight: 700;
    color: #1a1206;
    margin: 28px 0 8px 0;
    padding: 0;
    border: none;
    page-break-after: avoid;
}

h3 {
    font-family: Arial, sans-serif;
    font-size: 11.5pt;
    font-weight: 700;
    color: #3d3019;
    margin: 18px 0 6px 0;
    border-bottom: 1px solid #d4c9a8;
    padding-bottom: 4px;
    page-break-after: avoid;
}

h4, h5, h6 {
    font-family: Arial, sans-serif;
    font-weight: 700;
    color: #4a3c23;
    margin: 14px 0 4px 0;
    page-break-after: avoid;
}

p {
    margin: 6px 0 14px 0;
    text-align: justify;
    color: #2c2417;
}

/* Lists — block-level, proper spacing */
ul {
    display: block;
    margin: 8px 0 14px 0;
    padding-left: 28px;
    list-style-type: disc;
    color: #2c2417;
}
ol {
    display: block;
    margin: 8px 0 14px 0;
    padding-left: 28px;
    list-style-type: decimal;
    color: #2c2417;
}
li {
    display: list-item;
    margin-bottom: 5px;
    line-height: 1.7;
    color: #2c2417;
}
ul ul { list-style-type: circle; margin: 4px 0 4px 0; }
ul ul ul { list-style-type: square; }

blockquote {
    border-left: 4px solid #c4a84a;
    background-color: #f5efc6;
    margin: 14px 0;
    padding: 12px 18px;
    border-radius: 0 8px 8px 0;
    color: #3d3019;
    font-style: italic;
}
blockquote p {
    margin: 4px 0;
}

/* Tables */
table {
    width: 100%;
    border-collapse: collapse;
    margin: 14px 0;
    font-size: 10.5pt;
    page-break-inside: avoid;
    border: 2px solid #b8960f;
}
thead { background-color: #e6d270; }
th {
    background-color: #e6d270;
    color: #1a1206;
    font-weight: 700;
    padding: 10px 14px;
    border: 1px solid #b8960f;
    text-align: left;
}
td {
    padding: 9px 14px;
    border: 1px solid #d4c090;
    background-color: #fff8e1;
    color: #2c2417;
}
tr:nth-child(even) td {
    background-color: #ffedaa;
}

hr {
    border: none;
    height: 2px;
    background-color: #c4a84a;
    margin: 22px 0;
}

a { color: #8b6914; text-decoration: underline; }
strong { color: #1a1206; font-weight: 700; }
em { font-style: italic; }

/* Inline code */
code {
    font-family: Consolas, 'Courier New', monospace;
    font-size: 10pt;
}
p code, li code, td code, th code,
h2 code, h2 span code, h3 code, h4 code,
strong code, em code, a code,
blockquote code, dt code, dd code {
    background-color: #e8e0c8;
    color: #c7254e;
    padding: 2px 7px;
    border-radius: 4px;
    font-size: 10pt;
    border: 1px solid #d4c9a8;
}

/* Code blocks (Pygments output) */
.highlight {
    page-break-inside: avoid;
    margin: 12px 0 18px 0;
}
.highlight pre {
    background-color: #1e1e1e !important;
    color: #d4d4d4;
    border-radius: 8px;
    padding: 16px 20px;
    border-left: 4px solid #007acc;
    font-family: Consolas, 'Courier New', monospace;
    font-size: 10.5pt;
    line-height: 1.7;
    margin: 0;
    white-space: pre-wrap;
    word-wrap: break-word;
}

/* Standalone pre (no highlight wrapper) */
pre {
    background-color: #1e1e1e;
    color: #d4d4d4;
    border-radius: 8px;
    padding: 16px 20px;
    border-left: 4px solid #007acc;
    font-family: Consolas, 'Courier New', monospace;
    font-size: 10.5pt;
    line-height: 1.7;
    margin: 12px 0 18px 0;
    white-space: pre-wrap;
    word-wrap: break-word;
    page-break-inside: avoid;
}

img {
    max-width: 100%;
    height: auto;
    margin: 10px 0;
}

dt { font-weight: 700; color: #1a1206; margin-top: 10px; }
dd { margin-left: 20px; margin-bottom: 8px; color: #2c2417; }
"""

print("✅ Styling engine loaded")

✅ Styling engine loaded


In [ ]:
# ============================================================
#  🔨 Generate PDF from Markdown
# ============================================================

import os, platform
from pathlib import Path
from weasyprint import HTML
from IPython.display import display, HTML as IPHTML

# Detect environment and set output dir
if os.name == 'nt':
    OUTPUT_DIR = str(Path.home() / "Downloads")
elif os.path.isdir("/content"):
    OUTPUT_DIR = "/content"
else:
    OUTPUT_DIR = str(Path.home() / "Downloads")

os.makedirs(OUTPUT_DIR, exist_ok=True)

def md_to_styled_html(md_text):
    """Convert markdown to fully styled HTML with all inline styles."""
    html_body = markdown.markdown(
        md_text,
        extensions=[
            ListSpacingExtension(),            # auto blank lines before lists
            InlineCodeHighlightExtension(),    # fenced code → Pygments inline
            QuestionHighlightExtension(),      # h2 yellow highlight text only
            TableStyleExtension(),             # inline table/th/td styles
            ListStyleExtension(),              # inline ul/ol/li styles
            'tables',                          # | table | support
            'toc',                             # [TOC] support
            'sane_lists',                      # proper list handling
            'smarty',                          # smart quotes
            'fenced_code',                     # fallback for any missed fenced code
        ],
    )
    return f"""<!DOCTYPE html>
<html><head><meta charset="UTF-8"><style>{PDF_CSS}</style></head>
<body>{html_body}</body></html>"""

def generate_pdf(file_name, md_content):
    safe_name = re.sub(r'[^\w\s-]', '', file_name).strip().replace(' ', '_') or "document"
    pdf_path = os.path.join(OUTPUT_DIR, f"{safe_name}.pdf")

    styled_html = md_to_styled_html(md_content)

    # Debug: save HTML so you can inspect in browser
    html_path = pdf_path.replace('.pdf', '_debug.html')
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(styled_html)

    HTML(string=styled_html).write_pdf(pdf_path)
    size_kb = os.path.getsize(pdf_path) / 1024
    print(f"✅ PDF generated: {pdf_path} ({size_kb:.1f} KB)")
    print(f"   HTML debug: {html_path}")
    return pdf_path

pdf_file = generate_pdf(FILE_NAME, MD_CONTENT)

DEBUG:fontTools.ttLib.ttFont:Reading 'maxp' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'maxp' table
DEBUG:fontTools.subset.timer:Took 0.003s to load 'maxp'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'maxp'
INFO:fontTools.subset:maxp pruned
DEBUG:fontTools.ttLib.ttFont:Reading 'cmap' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'cmap' table
DEBUG:fontTools.ttLib.ttFont:Reading 'post' table from disk
DEBUG:fontTools.ttLib.ttFont:Decompiling 'post' table
DEBUG:fontTools.subset.timer:Took 0.010s to load 'cmap'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'cmap'
INFO:fontTools.subset:cmap pruned
INFO:fontTools.subset:fpgm dropped
INFO:fontTools.subset:prep dropped
INFO:fontTools.subset:cvt  dropped
INFO:fontTools.subset:kern dropped
DEBUG:fontTools.subset.timer:Took 0.000s to load 'post'
DEBUG:fontTools.subset.timer:Took 0.000s to prune 'post'
INFO:fontTools.subset:post pruned
INFO:fontTools.subset:GPOS dropped
INFO:fontTools.subset:GSUB dropped
DEBUG:f

✅ PDF generated: /content/test_pdf.pdf (118.6 KB)
   HTML debug: /content/test_pdf_debug.html


In [ ]:
# ============================================================
#  ⬇️ Download Button
# ============================================================

import base64

with open(pdf_file, "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

fname = os.path.basename(pdf_file)

display(IPHTML(f"""
<div style="text-align:center; margin:20px 0;">
    <a href="data:application/pdf;base64,{b64}" download="{fname}"
       style="
        display:inline-block; padding:14px 40px;
        background-color:#c4a84a; color:#fff;
        font-family:Arial,sans-serif; font-size:15px; font-weight:700;
        text-decoration:none; border-radius:8px;
        box-shadow:0 3px 10px rgba(0,0,0,0.15); cursor:pointer;
       ">⬇ Download {fname}</a>
</div>
"""))